# Algoritmos de Regresión

In [3]:
from pyspark.sql import SparkSession

# 1. Iniciamos Spark en este nuevo notebook
spark = SparkSession.builder.appName("Semana12_Supervisados").getOrCreate()

# 2. Leemos DIRECTAMENTE los datos que guardamos en el Paso 1
ruta_datos = "/home/jovyan/work/semanas/Semana 10/modelos/datos_etiquetados_kmeans"
df_clusters = spark.read.parquet(ruta_datos)

# 4. Mostramos la tabla para comprobar que TODO está ahí
df_clusters.select("marca", "precio_kg", "rating", "prediction").show(10)

+-----+------------------+-----------------+----------+
|marca|         precio_kg|           rating|prediction|
+-----+------------------+-----------------+----------+
|    0| 9.020000457763672|4.800000190734863|         1|
|    1| 6.860000133514404|4.699999809265137|         0|
|    1|16.579999923706055|4.699999809265137|         1|
|    3| 4.670000076293945|4.699999809265137|         0|
|    1|16.579999923706055|              5.0|         1|
|    3| 5.510000228881836|4.699999809265137|         0|
|    4|               7.0|4.699999809265137|         1|
|    4| 6.329999923706055|4.800000190734863|         1|
|    0| 6.800000190734863|4.699999809265137|         0|
|    3| 5.150000095367432|4.699999809265137|         0|
+-----+------------------+-----------------+----------+
only showing top 10 rows



In [4]:
from pyspark.ml.feature import VectorAssembler, StandardScaler
from pyspark.ml.regression import LinearRegression
from pyspark.ml.evaluation import RegressionEvaluator

In [5]:
# 1. Creamos el VectorAssembler (sin los precios)
assembler_regresion = VectorAssembler(
    inputCols=["rating", "opiniones"], 
    outputCol="features_regresion"
)
df_vector_reg = assembler_regresion.transform(df_clusters)

# 2. Escalamos las características
scaler_reg = StandardScaler(inputCol="features_regresion", outputCol="scaledFeatures_regresion")
scaler_model_reg = scaler_reg.fit(df_vector_reg)
df_para_regresion = scaler_model_reg.transform(df_vector_reg)

# 3. Renombramos la variable objetivo Y
df_para_regresion = df_para_regresion.withColumnRenamed("precio_kg", "label_precio")

# =======================================================================
# Borramos la columna prediction del K-Means 
# para que no choque con la de la Regresión Lineal
df_para_regresion = df_para_regresion.drop("prediction")
# =======================================================================

# 4. Dividimos en Entrenamiento (70%) y Prueba (30%)
train_reg, test_reg = df_para_regresion.randomSplit([0.7, 0.3], seed=42)

In [6]:
# Configurar el modelo de Regresión Lineal
lr_regresion = LinearRegression(
    featuresCol="scaledFeatures_regresion", 
    labelCol="label_precio", 
    maxIter=10
)

# Entrenar el modelo con los datos de entrenamiento
lr_reg_model = lr_regresion.fit(train_reg)

# Hacer las predicciones sobre los datos de prueba
predictions_regresion = lr_reg_model.transform(test_reg)

# Mostrar las predicciones junto al precio real
print("=== COMPARATIVA: PRECIO REAL VS PRECIO PREDICHO ===")
predictions_regresion.select("marca", "label_precio", "prediction").show(10)

=== COMPARATIVA: PRECIO REAL VS PRECIO PREDICHO ===
+-----+------------------+-----------------+
|marca|      label_precio|       prediction|
+-----+------------------+-----------------+
|    0|12.569999694824219|6.887154858970318|
|    0|12.569999694824219|6.887154858970318|
|    0|12.569999694824219|6.887154858970318|
|    0|12.569999694824219|6.887154858970318|
|    0|12.569999694824219|6.887154858970318|
|    0|12.569999694824219|6.887154858970318|
|    0|12.569999694824219|7.317322158572557|
|    0|12.569999694824219|7.317322158572557|
|    0|12.569999694824219|7.317322158572557|
|    0|12.569999694824219|7.317322158572557|
+-----+------------------+-----------------+
only showing top 10 rows



In [7]:
# Configurar los evaluadores de regresión
evaluator_r2 = RegressionEvaluator(labelCol="label_precio", predictionCol="prediction", metricName="r2")
evaluator_rmse = RegressionEvaluator(labelCol="label_precio", predictionCol="prediction", metricName="rmse")

r2 = evaluator_r2.evaluate(predictions_regresion)
rmse = evaluator_rmse.evaluate(predictions_regresion)

print("==================================================")
print("     EVALUACIÓN DE LA REGRESIÓN (SEMANA 12)       ")
print("==================================================")
print(f"R² (Coeficiente de Determinación): {r2 * 100:.2f}%")
print(f"RMSE (Error promedio del modelo):  {rmse:.4f}")
print("==================================================")

     EVALUACIÓN DE LA REGRESIÓN (SEMANA 12)       
R² (Coeficiente de Determinación): 5.68%
RMSE (Error promedio del modelo):  2.8363


In [8]:
# Imprimir la intersección (b) y los coeficientes (m)
print(f"Intersección (Precio base): {lr_reg_model.intercept:.4f}")
print(f"Coeficiente de 'rating':    {lr_reg_model.coefficients[0]:.4f}")
print(f"Coeficiente de 'opiniones': {lr_reg_model.coefficients[1]:.4f}")

Intersección (Precio base): -14.6993
Coeficiente de 'rating':    0.6410
Coeficiente de 'opiniones': 0.0421


### Ticket de Salida con sus datos

### 1. ¿El problema es el algoritmo o los datos?

En nuestro proyecto, el problema no está solamente en el algoritmo, sino principalmente en los datos disponibles. El modelo pudo ejecutarse y generar resultados, pero las variables que tenemos no explican por completo el comportamiento del CAE en los créditos hipotecarios.

En la etapa de clasificación, el algoritmo trabaja con variables como banco, región, monto, pie, dormitorios, tipo de interés y la tendencia del CAE. Sin embargo, el CAE no depende solo de esas variables internas del crédito, sino también de factores externos del mercado financiero.

Por eso, si el modelo no logra una predicción perfecta, no significa necesariamente que el algoritmo esté malo. Más bien, significa que el dataset todavía no tiene toda la información necesaria para representar completamente la realidad del mercado hipotecario.

### 2. ¿Cómo lo solucionarían en el mundo real sin hacer trampa?

En el mundo real, no sería correcto agregar directamente la respuesta que queremos predecir, como si el CAE subirá o no subirá, porque eso sería hacer trampa. Lo correcto sería mejorar el dataset original agregando variables que realmente influyen en el comportamiento del crédito hipotecario.

Para nuestro caso, faltarían datos importantes como la tasa de interés real ofrecida por cada banco, plazo del crédito, renta del solicitante, nivel de endeudamiento, historial crediticio, valor de la UF, inflación, tasa de política monetaria, riesgo del cliente, comuna específica del inmueble, antigüedad de la vivienda y condiciones comerciales de cada institución financiera.

Con esas variables, el modelo tendría más información para aprender patrones reales y entregar predicciones más confiables. En conclusión, el algoritmo funciona como herramienta, pero para mejorar el resultado se necesita un dataset más completo y más cercano a la realidad del mercado hipotecario.
